### Data Ingestion

In [ ]:
# from langchain_community.document_loaders import RecursiveUrlLoader

# import re
# from bs4 import BeautifulSoup

# def bs4_extractor(html: str) -> str:
#     soup = BeautifulSoup(html, "html.parser")
#     # Extract the main content of the page, for example, by getting the text from the body tag)
#      # remove noise
#     for tag in soup(["script", "style", "header", "footer", "nav"]):
#         tag.decompose()
    
#     # preserve table structure

#     for table in soup.find_all("table"):
#         rows = []
#         for row in table.find_all("tr"):
#             cells = [c.get_text(strip=True) for c in row.find_all(["td", "th"])]
#             rows.append(" | ".join(cells))
#         table.replace_with("\n".join(rows))
    
#     return re.sub(r"\n\n+", "\n\n", soup.get_text(separator="\n", strip=True)).strip()

# loader = RecursiveUrlLoader("https://reference.langchain.com/python/langchain-community/document_loaders/recursive_url_loader/RecursiveUrlLoader", max_depth=2, use_async=False, timeout=5, extractor=bs4_extractor, base_url="https://reference.langchain.com/python/langchain-community/document_loaders/recursive_url_loader/RecursiveUrlLoader")

# documents = []
# document_lazy = loader.lazy_load()

# for doc in document_lazy:
#     documents.append(doc)
# print(documents[0].page_content[:500])  # Print the first 500 characters of the first document
# print(f"Total documents loaded: {len(documents)}")
# print(documents[0].metadata)

In [ ]:
from langchain_community.document_loaders import FireCrawlLoader
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file

firecrawl_api_key = os.getenv("FIRECRAWL_API_KEY")

loader = FireCrawlLoader(
api_key=firecrawl_api_key,
url="https://www.udemy.com/course/advanced-rag-build-deploy-production-genai-apps/?couponCode=25BBPMXNVD35", 
mode = "crawl", params = {
    "limit": 10,
    "allowBackwardCrawling": False,
    "scrapeOptions": {
        "waitfor": 2000
    }
})

docs = []
docs_lazy = loader.lazy_load()

# async variant:
# docs_lazy = await loader.alazy_load()

for doc in docs_lazy:
    docs.append(doc)
    
print(docs[0].page_content[:500]) # Print the first 500 characters of the first document
print(docs[0].metadata)
print(f"Total documents loaded: {len(docs)}")

### Chunking

In [ ]:
### text splitting get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents,chunk_size = 1000,chunk_overlamp=200):
    """Split documents into smaller chunks for better processing and embedding generation."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlamp,
    length_function = len,
    separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of the chunk
    if split_docs:
        print("Example chunk:")
        print(f"content: {split_docs[0].page_content[:500]}")  # print the first 500 characters of the first chunk
        print(f"metadata: {split_docs[0].metadata}")
    return split_docs

In [ ]:
chunks = split_documents(docs)

### Embedding and VectorStoreDB

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer # embedding model is in sentence_transformers library
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
import os

In [30]:
class EmbeddingManager:
    """Manages the creation and storage of embeddings for documents."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

       """Initialize the embedding manager
       
       Args:
              model_name: Hugging Face model name for generating embeddings
       """
       self.model_name = model_name
       self.model = None # initially set to None, will be loaded when needed to initialize the model
       self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, Embedding dimension: {self.model.get_embedding_dimension()}") # print the dimension of the embeddings every text will have the same 384 dimension for this model
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
           
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts using the loaded model.
        
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initiialize the embedding manager and generate embeddings for the chunks
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9769.64it/s]


Model loaded successfully, Embedding dimension: 384


In [38]:
def sanitize_metadata(metadata: dict) -> dict:
    clean = {}
    for k, v in metadata.items():
        if v is None:
            clean[k] = ""
        elif isinstance(v, (str, int, float, bool)):
            clean[k] = v
        elif isinstance(v, list):
            clean[k] = ", ".join(str(i) for i in v)
        else:
            clean[k] = str(v)
    return clean

# After split_documents()
for chunk in chunks:
    chunk.metadata = sanitize_metadata(chunk.metadata)

# Now add to vector store — no more error

### VectorStore

In [39]:
class VectorStore:
    """Manages document embeddings in chromadb vector store."""

    def __init__(self, collection_name: str = "documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store and create a collection for storing document embeddings.
        
        Args:
            collection_name: Name of the collection to store document embeddings
            persist_directory: Directory where the vector store will be persisted
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize the chromadb client and collection to storing document embeddings."""
        # # Add this before get_or_create_collection
        # try:
        #     self.client.delete_collection(self.collection_name)
        #     print(f"Deleted existing collection: {self.collection_name}")
        # except:
        #     pass  # collection didn't exist, fine
        try:
            # create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # create or get collection for storing document embeddings
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata = {"description": "Collection for storing document embeddings for RAG",
        "hf:space": "cosine"})
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their corresponding embeddings to the vector store.
        
        Args:
            documents: List of document objects (e.g., langchain Document instances)
            embeddings: numpy array of embeddings corresponding to the documents, shape (len(documents), embedding_dim)
        """
        if not self.collection:
            raise ValueError("Vector store collection is not initialized.")
        
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to vector store...")
       
        #prepare data for chromadb
        ids = []
        metadatas = []
        document_texts = []
        embeddings_list = []
    
        for i,(doc, embedding) in enumerate(zip(documents, embeddings)):
           # Generate unique ID
           doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
           ids.append(doc_id)

           # Prepare metadata (you can customize this based on your document structure)
           metadata = dict(doc.metadata)
           metadata["doc_index"] = i
           metadata['content_length'] = len(doc.page_content)
           metadatas.append(metadata)

           # Document content
           document_texts.append(doc.page_content)

           # Embeddings
           embeddings_list.append(embedding.tolist())

         # Add to chromadb collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    
vector_store = VectorStore()
vector_store

Vector store initialized with collection: documents
existing documents in collection: 0


In [40]:
## convert the Text to embedding
texts = [doc.page_content for doc in chunks]

## Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store the documents and their embeddings in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 22 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.72it/s]

Generated embeddings with shape: (22, 384)
Adding 22 documents to vector store...
Successfully added 22 documents to vector store.
Total documents in collection after addition: 22


In [52]:
from turtle import distance


class RAGRetriever:
    """Handles query-based retrieval of relevant documents from the vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initialize the retriever with the vector store and embedding manager.
        
        Args:
            vector_store: Instance of the VectorStore class for retrieving document embeddings
            embedding_manager: Instance of the EmbeddingManager class for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents from the vector store based on the query.
        
        Args:
            query: The input query string for which to retrieve relevant documents
            top_k: The number of top relevant documents to return
            score_threshold: Minimum cosine similarity score for a document to be considered relevant
        Returns:
            A list of dictionaries containing the retrieved documents and their metadata"""
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        # Generate embedding for the query

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]  # shape (embedding_dim,)

        # Search in the vector store for relevant documents

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                # include=["metadatas", "documents", "embeddings"]
            )

            # process results to calculate cosine similarity scores
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    print(distance)
                    similarity_score = 1 - (distance / 2)  # convert distance to similarity score
                    # Add this inside loop to see actual values
                    print(f"Distance: {distance}, Similarity: {1-(distance/2)}")
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance': distance,
                            'range': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} relevant documents.")
            else:
                print("No documents retrieved for the query.")
            
            return retrieved_docs
        
        except Exception as e:
            print(f"Error occurred while retrieving documents: {e}")
            return []
        
retriever = RAGRetriever(vector_store, embedding_manager)
retriever

### Integration Vectordb Context pipeline with LLM output

In [51]:
from httpx import Response
from langchain_core.prompts import prompt
### simple rag pipline with Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()

### initialize the Groq LLM (set your GROQ_API_KEY in evn)
groq_api_key = os.getenv("groq_api_key")

llm = ChatGroq(model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## Simple rag function = retrieve context + generate response

def rag_simple(query,retriever,llm,top_k = 3):
    ## retrieve the context
    result = retriever.retrieve(query,top_k= top_k)
    context = "\n\n".join([doc['document'] for doc in result]) if result else ""
    if not context:
        return "No relevant context found to answer the question concisely"
    
    ## generate the answer using  GROQ LLM
    prompt = f"""use the following context to answer the question concisely.
        {context}

        Question: {query}

        Answer:"""
    
    # response = llm.invoke([prompt.format(context = context,query=query)])
    response = llm.invoke([HumanMessage(content = prompt)])
    return response.content


In [ ]:
# answer = rag_simple("what are his work exp  in 2024?",retriever,llm)
# print(answer)

Enhanced RAG Pipeline Features

In [49]:
from httpx import Response
from langchain_core.prompts import prompt
### simple rag pipline with Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Enhanced RAG pipeline Features
def rag_advanced(query, retriever, llm, top_k = 5, min_score= 0.2, return_context = False):
    """ RAG pipeline with extra features:
    - Returns answer, sources, confidence  score, and optionally full context."""
    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context':''}

    # prepare context and sources
    context = "\n\n".join([doc['document'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page','unknow'),
        'score': doc['similarity_score'],
        'preview': doc['document'][:300]+ '....'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f""" Use the following context to answer the question concisely. \ncontext:\n{context}\n\nQuestion:{query}\n\nAnswer:\n"""
    response = llm.invoke([HumanMessage(content = prompt)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [55]:
result = rag_advanced('how is the author ?', retriever, llm, top_k=3, min_score= 0.1, return_context= True)
print('Answer:',result['answer'])
print('sources:', result['sources'])
print('Confidence:', result['confidence'])
print('Context Preview:', result['context'][:300])

Retrieving documents for query: 'how is the author ?' with top_k=3 and score_threshold=0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 151.86it/s]

Generated embeddings with shape: (1, 384)
1.6244559288024902
Distance: 1.6244559288024902, Similarity: 0.18777203559875488
1.6723198890686035
Distance: 1.6723198890686035, Similarity: 0.16384005546569824
1.7534143924713135
Distance: 1.7534143924713135, Similarity: 0.12329280376434326
Retrieved 3 relevant documents.


Answer: The course is authored by **KGP Talkie (Laxmi Kant Tiwari)**.
sources: [{'source': 'unknown', 'page': 'unknow', 'score': 0.18777203559875488, 'preview': '![](https://img-c.udemycdn.com/course/480x270/7124685_10f6_4.jpg?w=3840&q=75)\n\nAdvanced RAG: Build & Deploy Production GenAI Apps\n\nBestseller\n\nHighest Rated\n\nRating: 4.6 out of 54.6 [(46 ratings)](https://www.udemy.com/course/advanced-rag-build-deploy-production-genai-apps/#reviews)\n\n348 students\n\n!....'}, {'source': 'unknown', 'page': 'unknow', 'score': 0.16384005546569824, 'preview': 'This is a hands-on, code-first course. Every section produces working, runnable code that you can adapt to your own documents and use cases.\n\n## Who this course is for:\n\n- Python developers who want to build production-grade RAG systems beyond basic tutorials\n- ML engineers looking to deploy LangCha....'}, {'source': 'unknown', 'page': 'unknow', 'score': 0.12329280376434326, 'preview': '- OpenAI-Compatible FastAPI Endpoints Exp